# Milestone 1: SME Portfolio Generation and Audit

## Objective

This notebook generates and audits a synthetic portfolio of 15,000 SME credit
applications. The portfolio will later be used to build:

1. A 12-month probability-of-default scorecard
2. Approve, refer and decline rules
3. A risk-based pricing engine
4. An early-warning monitoring system
5. A proactive credit-line-increase strategy

## Unit of observation

Each row represents one SME credit application at the application decision date.

## Target definition

`default_12m = 1` means that the applicant defaults within the 12 months following
the application date. Otherwise, `default_12m = 0`.

For a real lending portfolio, default would require a formally approved definition,
such as 90+ DPD, bankruptcy, or an unlikely-to-pay event.

## Limitation

All records are synthetic. The project demonstrates credit-risk methodology and
implementation, but the resulting model cannot be used for real lending decisions
without representative historical data and independent validation.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

SEED = 2026
N_APPLICANTS = 15_000

rng = np.random.default_rng(SEED)

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"
REPORTS_PATH = PROJECT_ROOT / "reports"

RAW_DATA_PATH.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Number of applicants:", f"{N_APPLICANTS:,}")
print("Random seed:", SEED)

Project root: /workspaces/SME-Lending-Decision-Analytics
Number of applicants: 15,000
Random seed: 2026


## 1. Generate applicant identifiers and categorical characteristics

The first group of variables describes the applicant rather than its financial
performance:

- `application_id`: unique identifier for each application
- `industry`: primary business activity
- `region`: geographical operating region
- `entity_type`: legal constitution of the business

The category probabilities are designed to create a mixed SME portfolio. They do
not represent an estimate of the actual Indian SME population.

At this stage, these variables do not determine whether a customer defaults.
Their risk effects, if any, will be introduced separately in the latent-risk
equation. Keeping data generation and risk generation separate makes the design
easier to audit.

In [2]:
application_id = np.array(
    [f"SME{number:06d}" for number in range(1, N_APPLICANTS + 1)]
)

industry_categories = [
    "Retail",
    "Manufacturing",
    "Professional Services",
    "Healthcare",
    "Hospitality",
    "Transport",
    "Construction",
    "Wholesale",
    "Technology",
    "Agriculture",
]

industry_probabilities = [
    0.18,
    0.14,
    0.14,
    0.08,
    0.08,
    0.10,
    0.10,
    0.08,
    0.06,
    0.04,
]

industry = rng.choice(
    industry_categories,
    size=N_APPLICANTS,
    p=industry_probabilities,
)

region_categories = [
    "North",
    "South",
    "East",
    "West",
    "Central",
]

region_probabilities = [
    0.27,
    0.25,
    0.18,
    0.25,
    0.05,
]

region = rng.choice(
    region_categories,
    size=N_APPLICANTS,
    p=region_probabilities,
)

entity_categories = [
    "Proprietorship",
    "Partnership",
    "LLP",
    "Private Limited",
    "Public Limited",
]

entity_probabilities = [
    0.38,
    0.22,
    0.12,
    0.24,
    0.04,
]

entity_type = rng.choice(
    entity_categories,
    size=N_APPLICANTS,
    p=entity_probabilities,
)

print("Applications generated:", len(application_id))
print("Unique application IDs:", len(np.unique(application_id)))

Applications generated: 15000
Unique application IDs: 15000


In [3]:
categorical_preview = pd.DataFrame(
    {
        "application_id": application_id,
        "industry": industry,
        "region": region,
        "entity_type": entity_type,
    }
)

print("Industry distribution:")
display(
    categorical_preview["industry"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("percentage")
    .to_frame()
)

print("Region distribution:")
display(
    categorical_preview["region"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("percentage")
    .to_frame()
)

print("Entity-type distribution:")
display(
    categorical_preview["entity_type"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("percentage")
    .to_frame()
)

Industry distribution:


,percentage
industry,
Retail,17.5700
Manufacturing,14.4700
Professional Services,13.9000
Transport,10.1500
Construction,10.0100
Hospitality,8.0500
Healthcare,7.7000
Wholesale,7.4900
Technology,6.4500


Region distribution:


,percentage
region,
North,27.5700
South,24.9500
West,24.6500
East,18.0400
Central,4.7900


Entity-type distribution:


,percentage
entity_type,
Proprietorship,38.3900
Private Limited,23.6500
Partnership,22.1900
LLP,12.1200
Public Limited,3.6500


## 2. Generate business size and financial characteristics

The next variables describe the applicant's operating history, business size,
profitability, indebtedness, liquidity and debt-servicing ability.

The variables are not generated independently:

- Employee count influences annual revenue.
- Annual revenue and profit margin determine net income.
- Total debt relative to revenue determines leverage.
- Profitability and leverage influence DSCR.

Creating economically connected variables produces a more realistic portfolio
than generating every feature independently.

Several distributions are right-skewed because most SME applicants are relatively
small, while a limited number are considerably larger.

In [4]:
years_in_business = np.clip(
    rng.gamma(
        shape=2.8,
        scale=3.0,
        size=N_APPLICANTS,
    ),
    0.5,
    40,
).round(1)

employees = np.clip(
    rng.lognormal(
        mean=np.log(8 + years_in_business),
        sigma=0.75,
        size=N_APPLICANTS,
    ),
    1,
    500,
).round().astype(int)

revenue_per_employee = rng.lognormal(
    mean=np.log(1_100_000),
    sigma=0.55,
    size=N_APPLICANTS,
)

annual_revenue = np.clip(
    employees * revenue_per_employee,
    500_000,
    500_000_000,
).round(-3)

print("Years in business:")
print(pd.Series(years_in_business).describe().round(2))

print("\nEmployees:")
print(pd.Series(employees).describe().round(2))

print("\nAnnual revenue:")
print(pd.Series(annual_revenue).describe().round(2))

Years in business:
count   15,000.0000
mean         8.4300
std          5.0600
min          0.5000
25%          4.7000
50%          7.5000
75%         11.1000
max         40.0000
dtype: float64

Employees:
count   15,000.0000
mean        21.7500
std         20.5800
min          1.0000
25%          9.0000
50%         16.0000
75%         27.0000
max        272.0000
dtype: float64

Annual revenue:
count        15,000.0000
mean     27,681,949.7300
std      34,154,033.7400
min         500,000.0000
25%       8,946,750.0000
50%      17,045,500.0000
75%      33,361,000.0000
max     500,000,000.0000
dtype: float64


In [5]:
industry_margin_adjustment = pd.Series(industry).map(
    {
        "Retail": -0.02,
        "Manufacturing": 0.00,
        "Professional Services": 0.04,
        "Healthcare": 0.03,
        "Hospitality": -0.01,
        "Transport": -0.01,
        "Construction": 0.00,
        "Wholesale": -0.025,
        "Technology": 0.05,
        "Agriculture": -0.015,
    }
).to_numpy()

revenue_growth_12m = np.clip(
    rng.normal(
        loc=0.08,
        scale=0.22,
        size=N_APPLICANTS,
    ),
    -0.60,
    1.50,
).round(4)

profit_margin = np.clip(
    0.08
    + industry_margin_adjustment
    + rng.normal(0, 0.06, N_APPLICANTS),
    -0.25,
    0.40,
).round(4)

net_income = (annual_revenue * profit_margin).round(-2)

debt_ratio = np.clip(
    rng.beta(
        a=2.2,
        b=3.0,
        size=N_APPLICANTS,
    ) * 1.60,
    0.02,
    1.60,
)

total_debt = (annual_revenue * debt_ratio).round(-3)

leverage = np.clip(
    total_debt / np.maximum(annual_revenue, 1),
    0,
    2,
).round(4)

current_ratio = np.clip(
    1.45
    + 1.20 * profit_margin
    - 0.25 * leverage
    + rng.normal(0, 0.45, N_APPLICANTS),
    0.20,
    5.00,
).round(2)

dscr = np.clip(
    1.55
    + 1.60 * profit_margin
    - 0.35 * leverage
    + 0.25 * revenue_growth_12m
    + rng.normal(0, 0.40, N_APPLICANTS),
    0.20,
    4.00,
).round(2)

financial_preview = pd.DataFrame(
    {
        "annual_revenue": annual_revenue,
        "revenue_growth_12m": revenue_growth_12m,
        "profit_margin": profit_margin,
        "net_income": net_income,
        "total_debt": total_debt,
        "leverage": leverage,
        "current_ratio": current_ratio,
        "dscr": dscr,
    }
)

financial_preview.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
annual_revenue,"15,000.0000","27,681,949.7330","34,154,033.7410","500,000.0000","8,946,750.0000","17,045,500.0000","33,361,000.0000","500,000,000.0000"
revenue_growth_12m,"15,000.0000",0.0830,0.2200,-0.6000,-0.0650,0.0810,0.2300,0.9870
profit_margin,"15,000.0000",0.0840,0.0650,-0.1880,0.0400,0.0840,0.1280,0.3310
net_income,"15,000.0000","2,329,493.9530","4,010,637.8740","-28,232,400.0000","411,650.0000","1,198,600.0000","2,842,425.0000","90,911,000.0000"
total_debt,"15,000.0000","18,823,375.0000","27,397,788.1750","86,000.0000","4,816,750.0000","10,448,000.0000","22,243,500.0000","551,058,000.0000"
leverage,"15,000.0000",0.6800,0.3190,0.0200,0.4350,0.6640,0.9100,1.5570
current_ratio,"15,000.0000",1.3810,0.4660,0.2000,1.0600,1.3800,1.6920,3.3600
dscr,"15,000.0000",1.4640,0.4260,0.2000,1.1800,1.4600,1.7500,3.1700


## 3. Generate credit-behaviour and application characteristics

The next variables describe the applicant's previous credit behaviour and the
current loan request.

The variables are generated with economically reasonable relationships:

- Higher utilisation and leverage increase the expected number of prior delinquencies.
- Negative profitability and repeated delinquency increase the probability of a
  public record.
- Higher utilisation, prior delinquency and public records reduce the bureau score.
- Credit-history length is related to years in business but is not exactly equal
  to it.
- Requested loan amount is linked to annual revenue.
- Collateral availability affects potential loss and policy decisions but will not
  directly determine the simulated default outcome in the first model.

In [6]:
utilization = np.clip(
    rng.beta(
        a=2.2,
        b=2.8,
        size=N_APPLICANTS,
    ),
    0,
    1,
).round(4)

delinquency_lambda = (
    0.15
    + 1.10 * utilization
    + 0.25 * (leverage > 1.00)
    + 0.20 * (profit_margin < 0)
)

prior_delinquencies_12m = np.clip(
    rng.poisson(delinquency_lambda),
    0,
    6,
).astype(int)

public_record_probability = np.clip(
    0.025
    + 0.060 * (prior_delinquencies_12m >= 2)
    + 0.050 * (profit_margin < 0),
    0.01,
    0.25,
)

public_records = rng.binomial(
    n=1,
    p=public_record_probability,
    size=N_APPLICANTS,
).astype(int)

credit_history_months = np.clip(
    years_in_business * 12
    + rng.normal(18, 24, N_APPLICANTS),
    3,
    360,
).round().astype(int)

bureau_score = np.clip(
    770
    - 125 * utilization
    - 18 * prior_delinquencies_12m
    - 55 * public_records
    + 7 * np.log1p(years_in_business)
    + rng.normal(0, 32, N_APPLICANTS),
    400,
    850,
).round().astype(int)

credit_preview = pd.DataFrame(
    {
        "utilization": utilization,
        "prior_delinquencies_12m": prior_delinquencies_12m,
        "public_records": public_records,
        "credit_history_months": credit_history_months,
        "bureau_score": bureau_score,
    }
)

credit_preview.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
utilization,"15,000.0000",0.4410,0.2020,0.0080,0.2820,0.4310,0.5890,0.9840
prior_delinquencies_12m,"15,000.0000",0.6970,0.8640,0.0000,0.0000,0.0000,1.0000,6.0000
public_records,"15,000.0000",0.0410,0.1980,0.0000,0.0000,0.0000,0.0000,1.0000
credit_history_months,"15,000.0000",119.2410,64.3060,3.0000,73.0000,109.0000,155.0000,360.0000
bureau_score,"15,000.0000",714.6200,48.3480,495.0000,683.0000,717.0000,749.0000,850.0000


In [7]:
requested_amount_ratio = rng.uniform(
    0.04,
    0.35,
    N_APPLICANTS,
)

requested_amount = np.clip(
    annual_revenue * requested_amount_ratio,
    100_000,
    50_000_000,
).round(-3)

collateral_probability = np.clip(
    0.40
    + 0.15 * (requested_amount_ratio >= 0.20)
    + 0.10 * np.isin(
        industry,
        ["Manufacturing", "Construction", "Transport", "Agriculture"],
    ),
    0.20,
    0.80,
)

collateral_flag = rng.binomial(
    n=1,
    p=collateral_probability,
    size=N_APPLICANTS,
).astype(int)

application_preview = pd.DataFrame(
    {
        "requested_amount": requested_amount,
        "requested_to_revenue": requested_amount / annual_revenue,
        "collateral_flag": collateral_flag,
    }
)

application_preview.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
requested_amount,"15,000.0000","5,287,555.9330","6,779,219.4590","100,000.0000","1,378,750.0000","2,997,000.0000","6,360,750.0000","50,000,000.0000"
requested_to_revenue,"15,000.0000",0.1950,0.0890,0.0400,0.1180,0.1940,0.2710,0.3500
collateral_flag,"15,000.0000",0.5170,0.5000,0.0000,0.0000,1.0000,1.0000,1.0000


## 5. Assemble the Applicant Portfolio

Each row represents one SME application at the application decision date.

The dataset contains:

- One unique applicant identifier
- Twenty observable applicant and credit-risk features
- No default outcome yet
- No hidden risk-model variables

Keeping the observable features separate from the future target-generation logic helps us prevent data leakage.

In [8]:
applicant_id = np.array(
    [f"SME{i:06d}" for i in range(1, N_APPLICANTS + 1)]
)

observable_features = [
    "industry",
    "region",
    "entity_type",
    "years_in_business",
    "employees",
    "annual_revenue",
    "revenue_growth_12m",
    "profit_margin",
    "net_income",
    "total_debt",
    "current_ratio",
    "dscr",
    "leverage",
    "bureau_score",
    "credit_history_months",
    "prior_delinquencies_12m",
    "utilization",
    "public_records",
    "requested_amount",
    "collateral_flag",
]

applicants = pd.DataFrame(
    {
        "applicant_id": applicant_id,
        "industry": industry,
        "region": region,
        "entity_type": entity_type,
        "years_in_business": years_in_business,
        "employees": employees,
        "annual_revenue": annual_revenue,
        "revenue_growth_12m": revenue_growth_12m,
        "profit_margin": profit_margin,
        "net_income": net_income,
        "total_debt": total_debt,
        "current_ratio": current_ratio,
        "dscr": dscr,
        "leverage": leverage,
        "bureau_score": bureau_score,
        "credit_history_months": credit_history_months,
        "prior_delinquencies_12m": prior_delinquencies_12m,
        "utilization": utilization,
        "public_records": public_records,
        "requested_amount": requested_amount,
        "collateral_flag": collateral_flag,
    }
)

print("Portfolio shape:", applicants.shape)
print("Number of applicants:", len(applicants))
print("Unique applicant IDs:", applicants["applicant_id"].nunique())
print("Observable feature count:", len(observable_features))

assert applicants.shape == (N_APPLICANTS, 21)
assert applicants["applicant_id"].is_unique
assert len(observable_features) == 20

applicants.head()

Portfolio shape: (15000, 21)
Number of applicants: 15000
Unique applicant IDs: 15000
Observable feature count: 20


,applicant_id,industry,region,entity_type,years_in_business,employees,annual_revenue,revenue_growth_12m,profit_margin,net_income,total_debt,current_ratio,dscr,leverage,bureau_score,credit_history_months,prior_delinquencies_12m,utilization,public_records,requested_amount,collateral_flag
0,SME000001,Retail,North,Proprietorship,8.4000,6,"8,127,000.0000",0.0293,0.0106,"86,100.0000","4,755,000.0000",1.4900,1.0800,0.5851,774,118,0,0.3226,0,"1,939,000.0000",1
1,SME000002,Transport,West,LLP,4.6000,19,"30,290,000.0000",-0.0745,0.1365,"4,134,600.0000","45,847,000.0000",1.3000,1.1400,1.5136,766,84,0,0.3914,0,"3,746,000.0000",1
2,SME000003,Healthcare,East,Proprietorship,7.6000,15,"6,503,000.0000",0.3306,-0.0230,"-149,600.0000","6,715,000.0000",1.4600,1.4400,1.0326,747,110,0,0.4214,0,"1,313,000.0000",1
3,SME000004,Professional Services,South,Private Limited,11.3000,33,"21,575,000.0000",0.4279,0.1615,"3,484,400.0000","10,114,000.0000",0.7500,1.0400,0.4688,699,158,0,0.5366,0,"7,482,000.0000",0
4,SME000005,Professional Services,South,Proprietorship,9.2000,17,"14,393,000.0000",-0.1981,0.1320,"1,899,900.0000","2,166,000.0000",1.7400,1.1900,0.1505,709,122,0,0.3827,0,"2,009,000.0000",0


## 5. Assemble the Applicant Portfolio

Each row represents one SME application at the application decision date.

The dataset contains:

- One unique applicant identifier
- Twenty observable applicant and credit-risk features
- No default outcome yet
- No hidden risk-model variables

Keeping the observable features separate from the future target-generation logic helps us prevent data leakage.

In [9]:
observable_features = [
    "industry",
    "region",
    "entity_type",
    "years_in_business",
    "employees",
    "annual_revenue",
    "revenue_growth_12m",
    "profit_margin",
    "net_income",
    "total_debt",
    "current_ratio",
    "dscr",
    "leverage",
    "bureau_score",
    "credit_history_months",
    "prior_delinquencies_12m",
    "utilization",
    "public_records",
    "requested_amount",
    "collateral_flag",
]

applicants = pd.DataFrame(
    {
        "applicant_id": applicant_id,
        "industry": industry,
        "region": region,
        "entity_type": entity_type,
        "years_in_business": years_in_business,
        "employees": employees,
        "annual_revenue": annual_revenue,
        "revenue_growth_12m": revenue_growth_12m,
        "profit_margin": profit_margin,
        "net_income": net_income,
        "total_debt": total_debt,
        "current_ratio": current_ratio,
        "dscr": dscr,
        "leverage": leverage,
        "bureau_score": bureau_score,
        "credit_history_months": credit_history_months,
        "prior_delinquencies_12m": prior_delinquencies_12m,
        "utilization": utilization,
        "public_records": public_records,
        "requested_amount": requested_amount,
        "collateral_flag": collateral_flag,
    }
)

print("Portfolio shape:", applicants.shape)
print("Number of applicants:", len(applicants))
print("Unique applicant IDs:", applicants["applicant_id"].nunique())
print("Observable feature count:", len(observable_features))

assert applicants.shape == (N_APPLICANTS, 21)
assert applicants["applicant_id"].is_unique
assert len(observable_features) == 20

applicants.head()

Portfolio shape: (15000, 21)
Number of applicants: 15000
Unique applicant IDs: 15000
Observable feature count: 20


,applicant_id,industry,region,entity_type,years_in_business,employees,annual_revenue,revenue_growth_12m,profit_margin,net_income,total_debt,current_ratio,dscr,leverage,bureau_score,credit_history_months,prior_delinquencies_12m,utilization,public_records,requested_amount,collateral_flag
0,SME000001,Retail,North,Proprietorship,8.4000,6,"8,127,000.0000",0.0293,0.0106,"86,100.0000","4,755,000.0000",1.4900,1.0800,0.5851,774,118,0,0.3226,0,"1,939,000.0000",1
1,SME000002,Transport,West,LLP,4.6000,19,"30,290,000.0000",-0.0745,0.1365,"4,134,600.0000","45,847,000.0000",1.3000,1.1400,1.5136,766,84,0,0.3914,0,"3,746,000.0000",1
2,SME000003,Healthcare,East,Proprietorship,7.6000,15,"6,503,000.0000",0.3306,-0.0230,"-149,600.0000","6,715,000.0000",1.4600,1.4400,1.0326,747,110,0,0.4214,0,"1,313,000.0000",1
3,SME000004,Professional Services,South,Private Limited,11.3000,33,"21,575,000.0000",0.4279,0.1615,"3,484,400.0000","10,114,000.0000",0.7500,1.0400,0.4688,699,158,0,0.5366,0,"7,482,000.0000",0
4,SME000005,Professional Services,South,Proprietorship,9.2000,17,"14,393,000.0000",-0.1981,0.1320,"1,899,900.0000","2,166,000.0000",1.7400,1.1900,0.1505,709,122,0,0.3827,0,"2,009,000.0000",0


In [10]:
quality_checks = {
    "Expected row count": len(applicants) == N_APPLICANTS,
    "Unique applicant IDs": applicants["applicant_id"].is_unique,
    "Exactly 20 observable features": len(observable_features) == 20,
    "No missing values": applicants.isna().sum().sum() == 0,
    "Utilization between 0 and 1": applicants["utilization"].between(0, 1).all(),
    "Bureau score between 400 and 850": applicants["bureau_score"].between(400, 850).all(),
    "DSCR is positive": applicants["dscr"].gt(0).all(),
    "Requested amount is positive": applicants["requested_amount"].gt(0).all(),
    "Public record flag is binary": applicants["public_records"].isin([0, 1]).all(),
    "Collateral flag is binary": applicants["collateral_flag"].isin([0, 1]).all(),
}

quality_results = pd.DataFrame(
    {
        "check": quality_checks.keys(),
        "passed": quality_checks.values(),
    }
)

display(quality_results)

assert quality_results["passed"].all(), "One or more quality checks failed."

print("All portfolio quality checks passed.")

,check,passed
0,Expected row count,True
1,Unique applicant IDs,True
2,Exactly 20 observable features,True
3,No missing values,True
4,Utilization between 0 and 1,True
5,Bureau score between 400 and 850,True
6,DSCR is positive,True
7,Requested amount is positive,True
8,Public record flag is binary,True
9,Collateral flag is binary,True


All portfolio quality checks passed.


## 6. Generate the 12-Month Default Outcome

Because this is synthetic data, there is no naturally observed default outcome. We therefore create a hidden credit-risk mechanism using financially meaningful relationships.

Expected relationships include:

- Lower DSCR increases default risk.
- Higher leverage and utilization increase risk.
- Prior delinquencies and public records increase risk.
- Higher bureau scores reduce risk.
- Strong liquidity and revenue growth reduce risk.
- Large loan requests relative to revenue increase risk.

The hidden risk equation is used only to generate the outcome. Its latent logit and true probability of default will not be included in the modelling dataset because doing so would cause target leakage.

In [11]:
# Use a separate random generator for the default outcome.
# This keeps target generation reproducible.
target_rng = np.random.default_rng(SEED + 1)

industry_risk_effect = applicants["industry"].map(
    {
        "Construction": 0.25,
        "Hospitality": 0.20,
        "Agriculture": 0.15,
        "Transport": 0.08,
        "Retail": 0.05,
        "Wholesale": 0.02,
        "Manufacturing": 0.00,
        "Professional Services": -0.08,
        "Healthcare": -0.10,
        "Technology": -0.15,
    }
)

requested_to_revenue = (
    applicants["requested_amount"] / applicants["annual_revenue"]
)

random_risk_noise = target_rng.normal(
    loc=0,
    scale=0.35,
    size=N_APPLICANTS,
)

latent_logit = (
    -1.75
    + 0.85 * ((1.40 - applicants["dscr"]) / 0.50)
    + 0.55 * ((applicants["leverage"] - 0.70) / 0.35)
    + 0.75 * ((applicants["utilization"] - 0.45) / 0.20)
    + 0.38 * applicants["prior_delinquencies_12m"]
    + 0.85 * applicants["public_records"]
    - 0.35 * ((applicants["bureau_score"] - 700) / 50)
    - 0.25 * ((applicants["current_ratio"] - 1.40) / 0.50)
    - 0.25 * (applicants["revenue_growth_12m"] / 0.25)
    + 0.35 * ((requested_to_revenue - 0.20) / 0.10)
    + industry_risk_effect
    + random_risk_noise
)

true_pd_12m = 1 / (1 + np.exp(-latent_logit))

applicants["default_12m"] = target_rng.binomial(
    n=1,
    p=true_pd_12m,
)

print(f"Observed 12-month default rate: {applicants['default_12m'].mean():.2%}")
print(f"Number of defaults: {applicants['default_12m'].sum():,}")
print(f"Number of non-defaults: {(applicants['default_12m'] == 0).sum():,}")

applicants["default_12m"].value_counts().rename(
    index={0: "Non-default", 1: "Default"}
)

Observed 12-month default rate: 23.02%
Number of defaults: 3,453
Number of non-defaults: 11,547


default_12m
Non-default    11547
Default         3453
Name: count, dtype: int64

## 7. Target and Directional Sanity Checks

The generated default outcome should satisfy two conditions:

1. The target must be binary and contain both defaults and non-defaults.
2. Defaulted applicants should, on average, exhibit weaker credit characteristics than non-defaulted applicants.

These checks validate the economic logic of the synthetic portfolio before model development.

In [12]:
target_checks = {
    "Target contains only 0 and 1":
        applicants["default_12m"].isin([0, 1]).all(),

    "Portfolio contains defaults":
        applicants["default_12m"].sum() > 0,

    "Portfolio contains non-defaults":
        (applicants["default_12m"] == 0).sum() > 0,

    "Default rate between 10% and 30%":
        applicants["default_12m"].mean() > 0.10
        and applicants["default_12m"].mean() < 0.30,
}

target_check_results = pd.DataFrame(
    {
        "check": target_checks.keys(),
        "passed": target_checks.values(),
    }
)

display(target_check_results)

assert target_check_results["passed"].all()
print("All target validation checks passed.")

,check,passed
0,Target contains only 0 and 1,True
1,Portfolio contains defaults,True
2,Portfolio contains non-defaults,True
3,Default rate between 10% and 30%,True


All target validation checks passed.


In [13]:
directional_analysis = (
    applicants
    .assign(
        requested_to_revenue=(
            applicants["requested_amount"]
            / applicants["annual_revenue"]
        )
    )
    .groupby("default_12m")
    .agg(
        applicant_count=("applicant_id", "count"),
        average_dscr=("dscr", "mean"),
        average_leverage=("leverage", "mean"),
        average_utilization=("utilization", "mean"),
        average_bureau_score=("bureau_score", "mean"),
        average_delinquencies=("prior_delinquencies_12m", "mean"),
        average_current_ratio=("current_ratio", "mean"),
        average_revenue_growth=("revenue_growth_12m", "mean"),
        average_requested_to_revenue=("requested_to_revenue", "mean"),
    )
    .rename(index={0: "Non-default", 1: "Default"})
    .round(3)
)

directional_analysis

,applicant_count,average_dscr,average_leverage,average_utilization,average_bureau_score,average_delinquencies,average_current_ratio,average_revenue_growth,average_requested_to_revenue
default_12m,,,,,,,,,
Non-default,11547,1.5280,0.6400,0.4050,723.0720,0.5820,1.4090,0.0940,0.1900
Default,3453,1.2520,0.8150,0.5590,686.3560,1.0840,1.2880,0.0460,0.2090


## 8. Export the Leakage-Safe Portfolio

The modelling dataset contains only:

- Applicant ID
- Twenty observable features available at application
- The observed 12-month default outcome

The hidden latent logit, true probability of default, and random risk noise are deliberately excluded. This prevents the future model from receiving information that would not be available during a genuine credit decision.

In [14]:
export_columns = (
    ["applicant_id"]
    + observable_features
    + ["default_12m"]
)

modelling_data = applicants[export_columns].copy()

forbidden_columns = {
    "latent_logit",
    "true_pd_12m",
    "random_risk_noise",
}

assert modelling_data.shape == (N_APPLICANTS, 22)
assert modelling_data["applicant_id"].is_unique
assert modelling_data["default_12m"].isin([0, 1]).all()
assert forbidden_columns.isdisjoint(modelling_data.columns)

raw_file_path = RAW_DATA_PATH / "sme_applicant_portfolio.csv"

modelling_data.to_csv(
    raw_file_path,
    index=False,
)

file_size_mb = raw_file_path.stat().st_size / (1024 ** 2)

print("Saved file:", raw_file_path)
print("Saved shape:", modelling_data.shape)
print(f"File size: {file_size_mb:.2f} MB")
print("Hidden risk variables excluded:", True)

Saved file: /workspaces/SME-Lending-Decision-Analytics/data/raw/sme_applicant_portfolio.csv
Saved shape: (15000, 22)
File size: 2.04 MB
Hidden risk variables excluded: True


In [15]:
reloaded_data = pd.read_csv(raw_file_path)

assert reloaded_data.shape == modelling_data.shape
assert reloaded_data["applicant_id"].is_unique
assert reloaded_data.isna().sum().sum() == 0
assert forbidden_columns.isdisjoint(reloaded_data.columns)

print("Reload test passed.")
print("Rows:", len(reloaded_data))
print("Columns:", len(reloaded_data.columns))

reloaded_data.head()

Reload test passed.
Rows: 15000
Columns: 22


,applicant_id,industry,region,entity_type,years_in_business,employees,annual_revenue,revenue_growth_12m,profit_margin,net_income,total_debt,current_ratio,dscr,leverage,bureau_score,credit_history_months,prior_delinquencies_12m,utilization,public_records,requested_amount,collateral_flag,default_12m
0,SME000001,Retail,North,Proprietorship,8.4000,6,"8,127,000.0000",0.0293,0.0106,"86,100.0000","4,755,000.0000",1.4900,1.0800,0.5851,774,118,0,0.3226,0,"1,939,000.0000",1,0
1,SME000002,Transport,West,LLP,4.6000,19,"30,290,000.0000",-0.0745,0.1365,"4,134,600.0000","45,847,000.0000",1.3000,1.1400,1.5136,766,84,0,0.3914,0,"3,746,000.0000",1,0
2,SME000003,Healthcare,East,Proprietorship,7.6000,15,"6,503,000.0000",0.3306,-0.0230,"-149,600.0000","6,715,000.0000",1.4600,1.4400,1.0326,747,110,0,0.4214,0,"1,313,000.0000",1,0
3,SME000004,Professional Services,South,Private Limited,11.3000,33,"21,575,000.0000",0.4279,0.1615,"3,484,400.0000","10,114,000.0000",0.7500,1.0400,0.4688,699,158,0,0.5366,0,"7,482,000.0000",0,0
4,SME000005,Professional Services,South,Proprietorship,9.2000,17,"14,393,000.0000",-0.1981,0.1320,"1,899,900.0000","2,166,000.0000",1.7400,1.1900,0.1505,709,122,0,0.3827,0,"2,009,000.0000",0,0


## 9–11. Reproduce historical booking, monthly behaviour and deterioration

The original Milestone 1 add-on is preserved in `src/milestone1_booking_and_panel.py`.
Run this cell after Sections 1–8 in a fresh kernel. It consumes the in-memory
`true_pd_12m`, `applicants`, `observable_features`, `SEED` and `RAW_DATA_PATH`.
The latent PD is intentionally excluded from exported modelling data.

The add-on uses independent random generators seeded with `SEED + 2` and
`SEED + 3` and creates `sme_booking_decision.csv`, `sme_onbook_portfolio.csv`
and `sme_behavioural_panel.csv`. It does not rewrite the applicant CSV.

**Target interpretation:** `deterioration_next_6_12mo` is a simulated binary
label based on latent PD, recent utilization, utilization change, leverage and
random noise. There are no observed months 25–36. This is an educational
simulation, not evidence of prediction against independently observed future outcomes.

For a non-destructive check against all four committed CSVs, run
`python scripts/reproduce_data.py` from the repository root. See
`docs/data_generation.md` for formulas, assumptions and limitations.


In [ ]:
from IPython.display import display

addon_path = PROJECT_ROOT / "src" / "milestone1_booking_and_panel.py"
if not addon_path.is_file():
    raise FileNotFoundError(f"Missing original generator: {addon_path}")

exec(compile(addon_path.read_text(encoding="utf-8"), str(addon_path), "exec"))
